In [39]:
import json
import os
import random
from itertools import chain
from collections import Counter

In [2]:
path_json = "/mnt/ssd/workspace/adi/repos/vh_deepfake_trainer/tools/dataset_assesor/result_taspen_photos.json"

In [3]:
with open(path_json, 'r') as f:
    # Parsing the JSON file into a Python dictionary
    data = json.load(f)

In [4]:
THRESHOLD_BLUR = 50
THRESHOLD_DARK = 50

In [5]:
data

{'../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/20.jpg': {'path': '../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/20.jpg',
  'status': 'FACE DETECTED',
  'pred_age': '40-49',
  'score_blur_face': 43.08879075456057,
  'score_dark': 4.485276823798955,
  'score_blur_img': 0,
  'score_dfd1-0-0': 0.0007176108192652464},
 '../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/2.3.jpg': {'path': '../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/2.3.jpg',
  'status': 'FACE DETECTED',
  'pred_age': '50-59',
  'score_blur_face': 23.336163460907073,
  'score_dark': 34.52542373160827,
  'score_blur_img': 0,
  'score_dfd1-0-0': 4.171665204921737e-05},
 '../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/1.6.jpg': {'path': '../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/1.6.jpg',
  'status': 'FACE DETECTED',
  'pred_age': '30-39',
  'score_blur_face': 26.81188485322734,
  'score_dark': 5.594511158475555,
  'score_blur_img': 0,
  'score_dfd1

In [6]:
data_filtered = {}
for path_now, d_now in data.items():
    # Filter on Blur
    if ((d_now['score_blur_face']<THRESHOLD_BLUR) & 
        (d_now['score_dark']<THRESHOLD_DARK)
    ):
        data_filtered[path_now] = d_now


In [7]:
print(len(data))
print(len(data_filtered))

51
48


### Create Pairing

Functions

In [44]:
def match(v, criteria):
    for key, rule in criteria.items():
        val = v.get(key)
        op = rule["op"]
        target = rule["value"]

        if op == "==":
            if val != target:
                return False
        elif op == "<=":
            if not (isinstance(val, (int, float)) and val <= target):
                return False
        elif op == ">=":
            if not (isinstance(val, (int, float)) and val >= target):
                return False
        else:
            raise ValueError(f"Unsupported operator: {op}")
    return True


def get_pair_key(key_source,dict_collection):

    list_all = [list(d.keys()) for d in dict_collection.values()]
    list_all = list(chain.from_iterable(list_all))
    # Count total frequency
    counter = Counter(list_all)
    max_count = max(counter.values())

    # Get *all* elements with same max frequency
    most_common_elements = [k for k, v in counter.items() if v == max_count]
    if key_source in most_common_elements:
        most_common_elements.remove(key_source)
    return random.choice(most_common_elements) if most_common_elements else None

def get_pair_candidates(list_criteria,data_all):
    dict_collection = {}
    for criteria_now in list_criteria:
        type_criteria = '-'.join(list(criteria_now.keys()))
        dict_collection[type_criteria] = {k: v for k, v in data_all.items() if 
                    match(v,criteria_now)}
    
    return dict_collection

In [51]:
data_completed = {}
swap_pair_list = []
for key_now, value_now in data_filtered.items():
    pred_age = value_now["pred_age"]
    list_criteria = [ 
    {"pred_age": {"op": "==", "value": pred_age}},
    {"score_blur_face": {"op": "<=", "value": 50}}
    ]
    dict_collection = get_pair_candidates(list_criteria,data_filtered)
    swap_pair = get_pair_key(key_now,dict_collection)
    swap_pair_list.append(swap_pair)
    value_now['swap_pair_path'] = swap_pair
    data_completed[key_now] = value_now

In [52]:
data_completed

{'../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/20.jpg': {'path': '../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/20.jpg',
  'status': 'FACE DETECTED',
  'pred_age': '40-49',
  'score_blur_face': 43.08879075456057,
  'score_dark': 4.485276823798955,
  'score_blur_img': 0,
  'score_dfd1-0-0': 0.0007176108192652464,
  'swap_pair_path': '../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/otentikasi_image.jpg'},
 '../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/2.3.jpg': {'path': '../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/2.3.jpg',
  'status': 'FACE DETECTED',
  'pred_age': '50-59',
  'score_blur_face': 23.336163460907073,
  'score_dark': 34.52542373160827,
  'score_blur_img': 0,
  'score_dfd1-0-0': 4.171665204921737e-05,
  'swap_pair_path': '../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/3.jpg'},
 '../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/1.6.jpg': {'path': '../../datasets/deepfake/vh_55plus/raw_ima

In [ ]:
#key_filter_now = "pred_age"
# Create list of filtered entries

list_criteria = [ 
{"pred_age": {"op": "==", "value": '40-49'}},
{"score_blur_face": {"op": "<=", "value": 50}}
]
for criteria_now in list_criteria:
    type_criteria = '-'.join(list(criteria_now.keys()))
    dict_collection[type_criteria] = {k: v for k, v in data_filtered.items() if 
                match(v,criteria_now)}

list_all = [list(d.keys()) for d in dict_collection.values()]
list_all = list(chain.from_iterable(list_all))

get_pair_key(list_all)

TypeError: get_pair_key() missing 1 required positional argument: 'all_elements'

In [31]:
#data_filtered[list(data_filtered.keys())[0]]

In [ ]:
# Randomly match between entry A and list B
#for data_now in data_filtered:
#entry_a = data_filtered[list(data_filtered.keys())[0]]
data_completed = {}
for key_now, value_now in data_filtered.items():
    # Run filtering with condition loop
    # condition 1
    #v.get("pred_age") == value_now["pred_age"]
    collection_b = {k: v for k, v in data_filtered.items() if v.get("pred_age") == value_now["pred_age"]}
    list_keys = list(collection_b.keys())
    list_keys.remove(key_now)
    value_now['swap_pair_path'] = random.choice(list_keys) if list_keys else None
    data_completed[key_now] = value_now

In [10]:
{k: v for k, v in data_completed.items() if v.get("swap_pair_path") is None}    

{'../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/fd4e0be5-aef3-4173-9b00-60a4de8a62f1 (1).jpg': {'path': '../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/fd4e0be5-aef3-4173-9b00-60a4de8a62f1 (1).jpg',
  'status': 'FACE DETECTED',
  'pred_age': '10-19',
  'score_blur_face': 37.079884673403804,
  'score_dark': 28.333684403918127,
  'score_blur_img': 4.32136269979963,
  'score_dfd1-0-0': 0.03761870041489601,
  'swap_pair_path': None},
 '../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/fd53947a-9563-4b4d-9d6b-1b1119e925cb (1).jpg': {'path': '../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/fd53947a-9563-4b4d-9d6b-1b1119e925cb (1).jpg',
  'status': 'FACE DETECTED',
  'pred_age': '20-29',
  'score_blur_face': 47.00920517517024,
  'score_dark': 19.527498966499223,
  'score_blur_img': 2.8769363371008443,
  'score_dfd1-0-0': 0.009782490320503712,
  'swap_pair_path': None}}

In [11]:
key_filter_now = "pred_age"
collection_b = {k: v for k, v in data_filtered.items() if v.get(key_filter_now) == value_now["pred_age"]}
list_keys = list(collection_b.keys())

In [12]:
criteria = {
    "age": {"op": "<=", "value": 30},
    "gender": {"op": "==", "value": "male"},
}

In [17]:
dict_collection

{'pred_age': True, 'score_blur_face': True}

In [24]:
from itertools import chain

In [30]:
from collections import Counter

# Combine all elements
all_elements = list_all

# Count total frequency
counter = Counter(all_elements)
max_count = max(counter.values())

# Get *all* elements with same max frequency
most_common_elements = [k for k, v in counter.items() if v == max_count]
if key_now in most_common_elements:
    most_common_elements.remove(key_now)
random.choice(most_common_elements) if most_common_elements else None

'../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/2.6.jpg'

In [27]:
most_common_elements

['../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/20.jpg',
 '../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/oten1.jpg',
 '../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/Oteen 1.jpg',
 '../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/oten_1.jpg',
 '../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/14.jpg',
 '../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/17.jpg',
 '../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/1.jpg',
 '../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/otentikasi_image.jpg',
 '../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/oten2.jpg',
 '../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/Oteen 2.jpg',
 '../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/2.6.jpg',
 '../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/3.3.jpg',
 '../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/f0161fc5-bb4a-4648-8d6b-adeeb914891e.jpg',
 '../../datasets/deep

In [32]:
key = random.choice(list(collection_b.keys()))
key

'../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/Oteen 2.jpg'

In [35]:
entry_a_complete =  entry_a
entry_a_complete["swap_pair_path"] = key
entry_a_complete

{'path': '../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/20.jpg',
 'status': 'FACE DETECTED',
 'pred_age': '40-49',
 'score_blur_face': 43.08879075456057,
 'score_dark': 4.485276823798955,
 'score_blur_img': 0,
 'score_dfd1-0-0': 0.0007176108192652464,
 'swap_pair_path': '../../datasets/deepfake/vh_55plus/raw_images/taspen_photos/Oteen 2.jpg'}

In [ ]:
"""

create dict of

{
source_path: "",
target_path: "",
output_path: "",
deepfake_method: ""
}
"""